last modified date : 2026.11

# GPU 메모리와 챗봇 실행 조건: Colab 실습


**오늘은 챗봇이 답하는 동안 무엇을 저장하고 계산하는지 살펴보고, 필요한 때 생성을 멈추는 방법을 확인합니다.** 모델을 올리고 인증을 붙이는 절차는 짧게 확인합니다. 학습의 중심은 메모리·캐시·생성 작업의 변화를 그림으로 설명하는 것입니다.

D52 챗봇 부록에는 `messages`로 대화를 받고 `/chat`으로 답하는 예제가 있었습니다. 그 부록을 하지 않았어도 시작할 수 있도록 **모델 연결·인증·스키마·API 파일을 이 노트에서 모두 만듭니다.** 이전 프로젝트와 Google Drive 마운트는 필요하지 않습니다.

Qwen2.5-1.5B-Instruct 한 개를 fp16으로 실행하고 같은 모델을 끝까지 재사용합니다. Streamlit 서버 대신 노트북의 대화 함수를 사용합니다. 화면 구현을 다시 배우지 않고 생성과 메모리의 관계에 집중합니다.

이론의 GPU·메모리 설명을 함께 읽습니다. RunPod 환경 운영 안내는 나중에 RunPod를 사용할 때 참고하고, 오늘의 준비·보관·종료는 이 노트의 Colab 절차를 따릅니다.


### 실행 환경과 오늘의 과제


**학습목표**

- [목표 1] GPU·모델 버전을 기록하고 가중치·KV 캐시·최대 메모리를 그래프로 구분할 수 있습니다.
- [목표 2] 같은 조건에서 캐시 사용 여부를 비교하고, 어텐션 그림에서 관찰한 값과 해석의 한계를 설명할 수 있습니다.
- [목표 3] 제공된 코드로 챗봇을 실행하고, 출력 소비 중단과 모델 생성 중단의 차이를 측정할 수 있습니다.

| 항목 | 사용할 환경 |
|---|---|
| 실행 | 각자의 Colab Pro+ GPU 런타임 하나. T4를 기준으로 범위를 구성하고 실제 GPU는 직접 확인 |
| 모델 | Qwen2.5-1.5B-Instruct, fp16, 커밋 해시 고정 |
| 파일 | `/content` 임시 디스크. 결과와 API 코드만 내려받기 |
| 챗봇 | 같은 런타임의 `/chat` API와 노트북 대화 함수 |
| 그림 | Plotly 그래프. 확대·범례 선택·어텐션 드롭다운 사용 |

각자 실행하고 4인 팀에서 예측과 해석을 비교합니다. 같은 노트북을 공유해도 GPU와 파일은 계정별로 따로입니다. GPU가 다른 팀원의 시간을 같은 조건의 속도로 비교하지 않습니다.

**시작 전:** 가중치가 그대로인데 문서가 길어지면 무엇이 늘어날지, 답변 읽기를 멈추면 모델도 멈출지 각자 예측합니다. 끝에서 실제 그래프를 근거로 처음의 설명을 고칩니다.

| 구간 | 배정 |
|---|---:|
| 도입·선수 지식 확인 | 20분 |
| 이론 1~3장과 Step 0~2 준비 | 60분 |
| 이론 4장과 Step 3 메모리·KV | 55분 |
| Step 4 길이·캐시 비교 | 40분 |
| Step 4의 어텐션 탐색 | 40분 |
| 이론 5장과 Step 5 오류 진단 | 30분 |
| 이론 6장과 Step 6 챗봇 실행 | 45분 |
| Step 7 실제 생성 중단 비교 | 35분 |
| Step 8 보관·종료와 개인 해석 | 35분 |
| **필수 학습 합계** | **360분** |

휴식 40분·점심 60분·설치 및 오류 대응 여유 20분을 포함해 480분으로 계획합니다. 실측 수업 시간은 아닙니다. 제공한 준비 코드의 모든 줄을 새로 작성하지 않고, 각 실험에서 바꾸는 조건과 읽을 그림을 먼저 확인합니다.


### Step 0. Colab GPU 런타임 준비


Colab에서 이 노트북을 열고 본인 Google Drive에 사본을 저장합니다. **런타임 설정에서 하드웨어 가속기를 GPU로 선택한 뒤 연결합니다.** T4가 제공되면 그 GPU로 시작하며, 이번 실습을 위해 A100으로 바꿀 필요는 없습니다.

Pro+에서도 할당되는 GPU와 사용 가능 시간은 달라질 수 있습니다. 다음 셀에서 GPU를 직접 확인하고, 팀원과 GPU가 다르면 속도 수치를 단순 비교하지 않습니다. GPU가 보이지 않으면 설치를 반복하기 전에 런타임 설정과 연결 상태를 확인합니다.

설치 셀은 처음 한 번 실행합니다. Colab이 제공한 GPU용 PyTorch는 다시 설치하지 않습니다. 설치 뒤에는 런타임의 **세션 다시 시작** 기능을 사용하고 다음 셀부터 실행합니다. 이때는 파일도 지우는 **연결 해제 및 런타임 삭제**를 선택하지 않습니다.


In [1]:
# Colab에 설치된 GPU용 torch는 유지합니다.
%pip install -q "transformers==5.17.0" "accelerate>=1.15,<2" "huggingface_hub>=1.33,<2" "fastapi==0.115.0" "uvicorn==0.30.6" requests "plotly>=6,<7" "numpy>=1.26,<3"


ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 2.7.1 requires fastapi<1,>=0.133, but you have fastapi 0.115.0 which is incompatible.
google-adk 2.7.1 requires starlette<2,>=1.3.1, but you have starlette 0.38.6 which is incompatible.
google-adk 2.7.1 requires uvicorn<1,>=0.34, but you have uvicorn 0.30.6 which is incompatible.
python-fasthtml 0.14.16 requires starlette>=1.0.1, but you have starlette 0.38.6 which is incompatible.
gradio 6.31.0 requires fastapi<1.0,>=0.115.2, but you have fastapi 0.115.0 which is incompatible.
gradio 6.31.0 requires starlette<2.0,>=1.0.1, but you have starlette 0.38.6 which is incompatible.


In [2]:
import os, sys, json, time, gc, re, shutil, subprocess, threading
from pathlib import Path
from datetime import datetime
from zoneinfo import ZoneInfo
from importlib.metadata import version
from packaging.version import Version

# 캐시 위치는 Hugging Face 라이브러리를 불러오기 전에 지정합니다.
os.environ["HF_HOME"] = "/content/hf"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
if sys.version_info < (3,10) or Version(torch.__version__.split("+")[0]) < Version("2.5"):
    raise RuntimeError("Python 3.10 이상·PyTorch 2.5 이상이 필요합니다. 강사와 Colab 런타임 버전을 확인하세요.")
if not torch.cuda.is_available():
    raise RuntimeError("GPU가 보이지 않습니다. Colab 런타임 설정의 GPU 선택과 연결 상태를 확인하세요.")
if not Path("/content").is_dir():
    raise RuntimeError("이 노트는 Colab의 /content 임시 저장공간을 사용합니다. Colab에서 실행하세요.")

from transformers import AutoConfig, AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import HfApi, snapshot_download

NOW = datetime.now(ZoneInfo("Asia/Seoul"))
RUN_ID = NOW.strftime("%Y%m%d-%H%M%S")
WORK = Path("/content/d53") / RUN_ID
RESULT_DIR = Path("/content/results/d53") / RUN_ID
MODEL_ROOT = Path("/content/models")
for folder in [WORK, RESULT_DIR, MODEL_ROOT]:
    folder.mkdir(parents=True, exist_ok=True)
os.chdir(WORK)

REPO = "Qwen/Qwen2.5-1.5B-Instruct"
GIB = 1024 ** 3
GPU_NAME = torch.cuda.get_device_name(0)
VERSIONS = {p: version(p) for p in ["torch", "transformers", "accelerate", "huggingface_hub", "fastapi", "uvicorn"]}
RESULTS = {"date":NOW.isoformat(), "run_id":RUN_ID,
           "runtime":{"platform":"Colab Pro+", "python":sys.version.split()[0], "cuda":torch.version.cuda},
           "gpu":{"name":GPU_NAME, "cc":list(torch.cuda.get_device_capability(0))},
           "model":{"repo":REPO, "precision":"fp16"}, "versions":VERSIONS,
           "execution_status":"GPU 실행 전", "bandwidth_gbs":None, "tps":None}
print("GPU:",GPU_NAME,"CUDA:",torch.version.cuda)
print("라이브러리:",VERSIONS)
print("임시 작업 폴더:",WORK,"결과 폴더:",RESULT_DIR)
print("/content의 파일은 런타임이 삭제되면 사라집니다. Step 8에서 결과와 그림을 내려받습니다.")


GPU: Tesla T4 CUDA: 13.0
라이브러리: {'torch': '2.11.0+cu130', 'transformers': '5.17.0', 'accelerate': '1.15.0', 'huggingface_hub': '1.33.0', 'fastapi': '0.115.0', 'uvicorn': '0.30.6'}
임시 작업 폴더: /content/d53/20261105-150928 결과 폴더: /content/results/d53/20261105-150928
/content의 파일은 런타임이 삭제되면 사라집니다. Step 8에서 결과와 그림을 내려받습니다.


In [3]:
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots

FIG_DIR=RESULT_DIR/"figures"
FIG_DIR.mkdir(exist_ok=True)
def show_save(fig,name):
    fig.update_layout(template="plotly_white",font=dict(family="Arial, Malgun Gothic, sans-serif",size=14),
                      title=dict(x=0.02,xanchor="left",font=dict(size=18)))
    # HTML과 같은 폴더에 plotly.min.js를 한 번 저장해 오프라인에서도 엽니다.
    fig.write_html(str(FIG_DIR/f"{name}.html"),include_plotlyjs="directory")
    fig.show(renderer="colab")
    return str(FIG_DIR/f"{name}.html")


### Step 1. GPU 상태와 메모리 범위 확인


[목표 1·2] 실행 전 GPU 상태를 기록합니다. 다음 셀의 전체·남은 양과 자신의 텐서·예약 사용량은 서로 범위가 다릅니다. 출력값을 읽고 **모델을 올리기 직전에 확인해야 하는 값**을 고릅니다.


In [4]:
subprocess.run(["nvidia-smi"], check=True)

def snapshot(label):
    torch.cuda.synchronize()
    free, total = torch.cuda.mem_get_info(0)
    return {"label": label, "free_gib": free/GIB, "total_gib": total/GIB,
            "allocated_gib": torch.cuda.memory_allocated(0)/GIB,
            "reserved_gib": torch.cuda.memory_reserved(0)/GIB,
            "peak_gib": torch.cuda.max_memory_allocated(0)/GIB}

START = snapshot("모델 로드 전")
RESULTS["gpu"]["total_gib"] = START["total_gib"]
print(json.dumps(START, ensure_ascii=False, indent=2))

# 먼저 예측을 말한 뒤 실행하세요. 해제 후 allocated와 reserved가 같을까요?
demo = torch.empty(64 * 1024 * 1024, dtype=torch.uint8, device="cuda:0")
print("텐서 생성:", snapshot("64MiB 생성"))
del demo
print("참조 해제:", snapshot("해제 후"))
torch.cuda.empty_cache()
print("빈 캐시 반환:", snapshot("empty_cache 후"))


{
  "label": "모델 로드 전",
  "free_gib": 14.46075439453125,
  "total_gib": 14.56317138671875,
  "allocated_gib": 0.0,
  "reserved_gib": 0.0,
  "peak_gib": 0.0
}
텐서 생성: {'label': '64MiB 생성', 'free_gib': 14.39825439453125, 'total_gib': 14.56317138671875, 'allocated_gib': 0.0625, 'reserved_gib': 0.0625, 'peak_gib': 0.0625}
참조 해제: {'label': '해제 후', 'free_gib': 14.39825439453125, 'total_gib': 14.56317138671875, 'allocated_gib': 0.0, 'reserved_gib': 0.0625, 'peak_gib': 0.0625}
빈 캐시 반환: {'label': 'empty_cache 후', 'free_gib': 14.46075439453125, 'total_gib': 14.56317138671875, 'allocated_gib': 0.0, 'reserved_gib': 0.0, 'peak_gib': 0.0625}


**기록 1:** 모델을 올리기 직전에 볼 값과 그 이유를 적습니다. 텐서를 해제한 뒤 실제 사용량과 예약한 양이 다르게 변했다면 관찰한 출력으로 설명합니다. 출력이 예상과 다르면 예상에 맞추기 위해 값을 고치지 않습니다.


### Step 2. 모델 버전 고정과 다운로드


[목표 1] 가중치 파일을 받기 전에 버전과 파일 크기를 확인합니다. **각자의 Colab 임시 디스크에 한 번 다운로드하고, 같은 런타임 안에서는 그 폴더를 재사용합니다.** 노트북을 공유했다고 모델 파일도 공유되는 것은 아닙니다.

`PINNED_REVISION`은 강사가 지정한 해시가 있으면 바꾸고, 없으면 조회한 해시를 결과에 남깁니다. 팀 결과를 비교하기 전에 GPU·모델 해시·dtype이 같은지 확인합니다. 모델을 Google Drive에 복사하거나 별도로 마운트하지 않아도 진행할 수 있습니다.


In [5]:
PINNED_REVISION = None  # 강사가 지정한 전체 커밋 해시가 있으면 문자열로 입력하세요.
api = HfApi()
info = api.model_info(REPO, revision=PINNED_REVISION, files_metadata=True)
REVISION = info.sha
MODEL_DIR = MODEL_ROOT / REPO.split("/")[-1].lower() / REVISION[:12]

def keep_model_file(name):
    return not (name.endswith((".bin", ".pt", ".pth", ".gguf", ".onnx")) or name.startswith("original/"))

FILES = [{"name": f.rfilename, "bytes": f.size} for f in info.siblings if keep_model_file(f.rfilename)]
if not FILES or any(f["bytes"] is None for f in FILES):
    raise RuntimeError("파일 크기를 확인하지 못했습니다. 조회 결과와 연결을 확인하세요.")
assert any(f["name"].endswith(".safetensors") for f in FILES), "safetensors 가중치가 있는지 확인하세요."
DOWNLOAD_BYTES = sum(f["bytes"] for f in FILES)
FREE_DISK = shutil.disk_usage(MODEL_ROOT).free
print("모델:", REPO, "커밋:", REVISION)
print(f"파일 합계 {DOWNLOAD_BYTES/GIB:.2f}GiB / 디스크 여유 {FREE_DISK/GIB:.2f}GiB")
print("모델 폴더:", MODEL_DIR)
for f in FILES:
    if f["name"].endswith(".safetensors"):
        print(f["name"], f"{f['bytes']/GIB:.2f}GiB")
if DOWNLOAD_BYTES > FREE_DISK * 0.8:
    raise RuntimeError("전체 파일을 새로 받아도 충분한 여유가 있는지 강사와 확인하세요.")


모델: Qwen/Qwen2.5-1.5B-Instruct 커밋: 989aa7980e4cf806f80c7fef2b1adb7bc71aa306
파일 합계 2.89GiB / 디스크 여유 235.34GiB
모델 폴더: /content/models/qwen2.5-1.5b-instruct/989aa7980e4c
model.safetensors 2.88GiB


In [6]:
MODEL_DIR.mkdir(parents=True, exist_ok=True)
t0 = time.perf_counter()
snapshot_download(REPO, revision=REVISION, local_dir=MODEL_DIR,
                  allow_patterns=[f["name"] for f in FILES])
missing = [f["name"] for f in FILES if not (MODEL_DIR/f["name"]).is_file()]
if missing:
    raise RuntimeError(f"필요한 파일이 없습니다: {missing}")
MANIFEST = {"repo": REPO, "revision": REVISION, "files": FILES,
            "local_dir": str(MODEL_DIR), "download_seconds": time.perf_counter()-t0}
(RESULT_DIR/"model_manifest.json").write_text(json.dumps(MANIFEST, ensure_ascii=False, indent=2), encoding="utf-8")
RESULTS["model"].update(revision=REVISION, local_dir=str(MODEL_DIR))
print("다운로드와 파일 존재 확인 완료. 버전 기록을 저장했습니다.")


Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 10 files:   0%|          | 0/10 [00:00<?, ?it/s]

다운로드와 파일 존재 확인 완료. 버전 기록을 저장했습니다.


**기록 2:** 모델 이름, 전체 커밋 해시, 모델 폴더, 파일 크기 합계를 적습니다. 다른 팀과 답이 달랐을 때 같은 버전인지 확인할 수 있어야 합니다. 파일이 있다는 검사는 파일 내용의 무결성을 독립적으로 검증한 검사와 구분합니다.


### Step 3. 가중치와 KV 캐시 직접 측정


[목표 1] **가중치는 로드 전후의 차이, KV 캐시는 보관 전후의 차이, 생성 중 사용량은 최대값으로 봅니다.** 숫자의 측정 시점이 다르므로 서로 바꾸어 부르지 않습니다. 먼저 가중치를 계산하고 모델을 한 번만 올립니다.


In [7]:
before = snapshot("로드 전")
t0 = time.perf_counter()
tok = AutoTokenizer.from_pretrained(MODEL_DIR, local_files_only=True)
model = AutoModelForCausalLM.from_pretrained(
    MODEL_DIR, local_files_only=True, dtype=torch.float16,
    device_map={"": 0}, attn_implementation="sdpa").eval()
torch.cuda.synchronize()
LOAD_SECONDS = time.perf_counter()-t0
cfg = model.config
after = snapshot("로드 후")
PARAMS = sum(p.numel() for p in model.parameters())
W_CALC_GIB = PARAMS * 2 / GIB
W_REAL_GIB = after["allocated_gib"]-before["allocated_gib"]

DEVICE_MAP = getattr(model, "hf_device_map", None)
if DEVICE_MAP is None:
    DEVICE_MAP = {"": str(next(model.parameters()).device)}

print("모델 장치:", DEVICE_MAP)


print(f"파라미터 {PARAMS:,}개 / 단순 계산 {W_CALC_GIB:.3f}GiB / 추가 텐서 사용량 {W_REAL_GIB:.3f}GiB")
RESULTS["load"] = {"params": PARAMS, "dtype": str(model.dtype), "device_map": DEVICE_MAP,
                   "w_calc_gb": W_CALC_GIB*GIB/1e9, "w_real_gb": W_REAL_GIB*GIB/1e9,
                   "w_calc_gib": W_CALC_GIB, "w_real_gib": W_REAL_GIB,
                   "seconds": LOAD_SECONDS, "legacy_gb_keys_unit": "GB", "measured": True}

def kv_bytes(cfg, tokens, requests=1):
    kv_heads = getattr(cfg, "num_key_value_heads", cfg.num_attention_heads)
    head_dim = getattr(cfg, "head_dim", None) or cfg.hidden_size//cfg.num_attention_heads
    return 2*cfg.num_hidden_layers*kv_heads*head_dim*tokens*requests*2

print(f"fp16 KV 캐시의 단순 추정: token 하나에 {kv_bytes(cfg,1)/1024:.1f}KiB")


Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

모델 장치: {'': 'cuda:0'}
파라미터 1,543,714,304개 / 단순 계산 2.875GiB / 추가 텐서 사용량 2.875GiB
fp16 KV 캐시의 단순 추정: token 하나에 28.0KiB


In [8]:
SYSTEM = "당신은 한국어로 간단명료하게 답하는 안내자입니다. 제공된 문서에 없는 내용은 확인할 수 없다고 답하세요."
DOC = "실습용 안내문: 교육장 이용 시간은 오전 9시부터 오후 6시까지입니다."
QUESTION = "교육장은 몇 시에 문을 닫습니까?"

def make_inputs(user_text):
    messages = [{"role":"system", "content":SYSTEM}, {"role":"user", "content":user_text}]
    text = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    return tok(text, return_tensors="pt").to(model.device)

def measure_generation(user_text, max_new_tokens=32):
    inputs = make_inputs(user_text)
    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    with torch.inference_mode():
        output = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False,
                                pad_token_id=tok.eos_token_id)
    torch.cuda.synchronize()
    elapsed = time.perf_counter()-t0
    n = inputs["input_ids"].shape[1]
    result = {"input_tokens":n, "output_tokens":output.shape[1]-n,
              "max_new_tokens":max_new_tokens, "seconds":elapsed,
              "peak_gib":torch.cuda.max_memory_allocated(0)/GIB,
              "response":tok.decode(output[0,n:], skip_special_tokens=True)}
    del inputs, output
    return result

FIRST = measure_generation(DOC+"\n"+QUESTION)
print(json.dumps(FIRST, ensure_ascii=False, indent=2))
RESULTS["first_request"] = FIRST


{
  "input_tokens": 87,
  "output_tokens": 16,
  "max_new_tokens": 32,
  "seconds": 2.726881642999956,
  "peak_gib": 2.892026424407959,
  "response": "교육장의 문은 오후 6시에 닫습니다."
}


**첫 응답 확인:** 입력 token 수, 생성 중 최대 메모리, 응답을 읽습니다. 첫 실행에는 초기화의 영향이 포함될 수 있습니다. 응답은 실습용 안내문과 대조하며, 문서에 없는 규정을 모델이 말해도 사실로 사용하지 않습니다.

**KV 캐시를 남겨서 측정하기**

생성이 끝나면 캐시도 함수 안에서 해제되므로, 종료 뒤의 `memory_allocated()`만 봐서는 캐시가 얼마나 늘었는지 알기 어렵습니다. 이번에는 입력을 한 번 읽는 prefill을 수행하고, 반환된 `past_key_values`를 잠시 보관합니다. 출력 객체의 다른 값은 해제한 뒤 메모리 차이를 기록합니다.

다음 준비 함수는 대화의 역할 구분과 질문 끝을 남겨 두고, 중간의 참고 문장으로 token 수를 맞춥니다. 이 입력은 메모리 비교용이며, 답변 품질을 평가하는 문서가 아닙니다.


In [9]:
def fixed_length_inputs(n_tokens):
    messages = [{"role":"system", "content":SYSTEM},
                {"role":"user", "content":DOC+"\n<<FILL>>\n"+QUESTION}]
    template = tok.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    head, tail = template.split("<<FILL>>")
    head_ids = tok(head, add_special_tokens=False)["input_ids"]
    tail_ids = tok(tail, add_special_tokens=False)["input_ids"]
    filler = tok(" 참고 자료입니다.", add_special_tokens=False)["input_ids"]
    extra = n_tokens-len(head_ids)-len(tail_ids)
    if extra < 0:
        raise ValueError("기본 입력보다 긴 길이를 선택하세요.")
    ids = head_ids + (filler*((extra//len(filler))+1))[:extra] + tail_ids
    ids = torch.tensor([ids], device=model.device)
    return {"input_ids":ids, "attention_mask":torch.ones_like(ids)}


In [10]:
KV_ROWS = []
for n in [256, 1024]:
    gc.collect(); torch.cuda.empty_cache()
    inputs = fixed_length_inputs(n)
    torch.cuda.synchronize()
    base = torch.cuda.memory_allocated(0)
    cache = output = None
    try:
        # 마지막 위치의 logits만 반환해 큰 출력 텐서가 남는 것을 피합니다.
        with torch.inference_mode():
            output = model(**inputs, use_cache=True, logits_to_keep=1)
        cache = output.past_key_values
        if cache is None:
            raise RuntimeError("KV 캐시가 반환되지 않았습니다. 모델 설정과 실행 출력을 확인하세요.")
        del output
        torch.cuda.synchronize()
        held = torch.cuda.memory_allocated(0)
        row = {"input_tokens": n, "before_allocated_gib":base/GIB,
               "held_allocated_gib":held/GIB, "kv_calc_mib": kv_bytes(cfg,n)/1024**2,
               "retained_delta_mib": (held-base)/1024**2, "oom": None}
    except torch.cuda.OutOfMemoryError as error:
        row = {"input_tokens": n, "kv_calc_mib": kv_bytes(cfg,n)/1024**2,
               "retained_delta_mib": None, "oom": str(error)}
    finally:
        # 캐시와 입력을 해제한 다음 별도의 상태도 남깁니다.
        cache = output = None
        del inputs
        gc.collect(); torch.cuda.empty_cache()
    row["after_release_allocated_gib"] = snapshot("KV 해제 후")["allocated_gib"]
    KV_ROWS.append(row)
    print(row)
    if row["oom"]:
        print("추가 측정을 멈추고 오류와 현재 GPU 여유를 확인합니다.")
        break
RESULTS["kv_cache"] = KV_ROWS


{'input_tokens': 256, 'before_allocated_gib': 2.884307861328125, 'held_allocated_gib': 2.891143798828125, 'kv_calc_mib': 7.0, 'retained_delta_mib': 7.0, 'oom': None, 'after_release_allocated_gib': 2.8843040466308594}
{'input_tokens': 1024, 'before_allocated_gib': 2.884319305419922, 'held_allocated_gib': 2.911663055419922, 'kv_calc_mib': 28.0, 'retained_delta_mib': 28.0, 'oom': None, 'after_release_allocated_gib': 2.8843040466308594}


**기록 3:** 가중치의 계산값과 로드 후 추가 사용량, 256·1,024token의 KV 계산값과 캐시 보관 중 추가 할당량을 각각 비교합니다. `before`·`held`·`after_release`의 GiB 값을 보고 보관·해제 시점을 설명합니다. 마지막 해제에는 입력 텐서도 포함됩니다.

`retained_delta_mib`는 **캐시를 보관한 상태의 추가 할당량**입니다. 내부 버퍼 등도 영향을 줄 수 있으므로 캐시 파일 크기를 직접 잰 값이라고 단정하지 않습니다. Qwen2.5-1.5B의 fp16 KV 계산값은 token당 28KiB이지만, 자기 측정값을 이 값에 맞춰 고치지 않습니다.

**계산 조건 확인:** `kv_bytes()`는 모델의 층 수·KV 헤드 수·헤드 차원을 읽고, 현재 fp16의 2byte를 곱합니다. 다른 모델이나 저장 형식을 계산할 때는 이 조건부터 확인합니다. 대형 모델의 파일을 추가로 받아야 확인할 수 있는 개념은 아닙니다.


In [11]:
memory_fig=make_subplots(rows=1,cols=2,subplot_titles=("가중치: 로드 전후 차이", "KV: 입력 길이별 보관 중 추가량"))
memory_fig.add_trace(go.Bar(x=["계산","추가 할당 실측"],y=[W_CALC_GIB,W_REAL_GIB],name="가중치"),1,1)
valid=[r for r in KV_ROWS if r["retained_delta_mib"] is not None]
for key,label in [("kv_calc_mib","KV 계산"),("retained_delta_mib","보관 중 추가 할당")]:
    memory_fig.add_trace(go.Bar(x=[str(r["input_tokens"])+"token" for r in valid],
                                y=[r[key] for r in valid],name=label),1,2)
memory_fig.update_yaxes(title_text="GiB",rangemode="tozero",row=1,col=1)
memory_fig.update_yaxes(title_text="MiB",rangemode="tozero",row=1,col=2)
memory_fig.update_layout(height=430,barmode="group",title="같은 모델에서도 측정 시점과 단위가 다릅니다")
show_save(memory_fig,"01_weights_and_kv")


'/content/results/d53/20261105-150928/figures/01_weights_and_kv.html'

**그림 읽기:** 두 패널의 단위가 다른 이유를 설명합니다. 입력 길이가 늘어도 왼쪽의 가중치가 함께 늘어야 하는지 판단하고, 오른쪽의 계산·실측 차이를 캐시 자체의 크기 차이라고 단정할 수 있는지 씁니다.


### Step 4. 입력 길이·캐시·어텐션 비교


[목표 1·2] **길이별 최대 메모리를 짧게 확인한 뒤, 캐시를 사용하는 이유를 직접 비교합니다.** D38의 길이 비교 활동을 했다면 첫 표는 복습입니다. 이번에는 같은 입력에서 `use_cache` 하나를 바꾸고, 캐시를 보관하는 비용과 재계산 시간을 함께 봅니다.


In [12]:
CURVE = []
for n in [256, 512, 1024]:
    inputs = fixed_length_inputs(n)
    torch.cuda.reset_peak_memory_stats(0)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    output = None
    try:
        with torch.inference_mode():
            output = model.generate(**inputs, min_new_tokens=8, max_new_tokens=8,
                                    do_sample=False, pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        row = {"tokens":n, "output_tokens":output.shape[1]-n,
               "peak_gib":torch.cuda.max_memory_allocated(0)/GIB,
               "kv_calc_gib":kv_bytes(cfg,n+8)/GIB,
               "seconds":time.perf_counter()-t0, "oom":None}
    except torch.cuda.OutOfMemoryError as error:
        row = {"tokens":n, "output_tokens":None, "oom":str(error), "peak_gib":None}
    finally:
        output = None
        del inputs
        gc.collect(); torch.cuda.empty_cache()
    CURVE.append(row)
    print(row)
    if row["oom"]:
        print("추가 측정을 멈추고 오류와 현재 여유를 확인합니다.")
        break
RESULTS["input_curve"] = CURVE
RESULTS["calc_vs_real"] = [{"자리":"가중치", "계산":W_CALC_GIB*GIB/1e9, "실측":W_REAL_GIB*GIB/1e9}]


{'tokens': 256, 'output_tokens': 8, 'peak_gib': 2.907388210296631, 'kv_calc_gib': 0.007049560546875, 'seconds': 0.3414316550001786, 'oom': None}
{'tokens': 512, 'output_tokens': 8, 'peak_gib': 2.9441418647766113, 'kv_calc_gib': 0.013885498046875, 'seconds': 0.3807494680002037, 'oom': None}
{'tokens': 1024, 'output_tokens': 8, 'peak_gib': 3.0596413612365723, 'kv_calc_gib': 0.027557373046875, 'seconds': 0.5673143450001135, 'oom': None}


In [13]:
length_rows=[r for r in CURVE if r.get("peak_gib") is not None]
length_fig=go.Figure(go.Scatter(x=[r["tokens"] for r in length_rows],
    y=[r["peak_gib"] for r in length_rows],mode="lines+markers",name="생성 중 최대값"))
length_fig.update_layout(title="입력만 늘렸을 때의 최대 텐서 메모리: 출력 8token",
                         xaxis_title="입력 token 수",yaxis_title="최대 메모리 GiB",height=400)
show_save(length_fig,"02_length_peak")


'/content/results/d53/20261105-150928/figures/02_length_peak.html'

**캐시 비교에서 고정할 조건**

| 고정 | 바꿀 것 | 기록할 것 |
|---|---|---|
| 같은 GPU·모델·fp16·입력 512token·출력 32token | KV 캐시 사용 / 미사용 | 시간·최대 메모리·실제 출력 token 수 |

캐시는 이전 token의 키와 값을 재사용해 재계산을 줄입니다. 캐시를 끄면 저장량만 줄고 모든 사용량이 줄 것이라고 미리 결론 내리지 않습니다. 다시 계산하는 중간값도 최대 메모리에 영향을 줍니다.

두 조건을 한 번씩 예열하고 순서를 바꿔 두 번씩 측정합니다. 출력 32token을 맞추기 위해 이 비교에서만 최소 생성 길이도 지정합니다. 답변 품질 시험이 아닌 계산 부하 비교입니다.


In [14]:
def cache_run(use_cache):
    inputs=fixed_length_inputs(512)
    output=None
    torch.cuda.reset_peak_memory_stats(0);torch.cuda.synchronize()
    started=time.perf_counter()
    try:
        with torch.inference_mode():
            output=model.generate(**inputs,use_cache=use_cache,min_new_tokens=32,max_new_tokens=32,
                                  do_sample=False,pad_token_id=tok.eos_token_id)
        torch.cuda.synchronize()
        return {"use_cache":use_cache,"input_tokens":inputs["input_ids"].shape[1],
                "output_tokens":output.shape[1]-inputs["input_ids"].shape[1],
                "seconds":time.perf_counter()-started,"peak_gib":torch.cuda.max_memory_allocated(0)/GIB}
    finally:
        output=None
        del inputs
        gc.collect();torch.cuda.empty_cache()

for flag in [False,True]:cache_run(flag)
CACHE_ROWS=[cache_run(flag) for flag in [False,True,True,False]]
RESULTS["cache_ablation"]={"warmup_per_condition":1,"runs":CACHE_ROWS}
cache_fig=make_subplots(rows=1,cols=2,subplot_titles=("생성 시간", "생성 중 최대 메모리"))
for col,key,unit in [(1,"seconds","초"),(2,"peak_gib","GiB")]:
    for flag,label in [(False,"캐시 끔"),(True,"캐시 켬")]:
        values=[r[key] for r in CACHE_ROWS if r["use_cache"]==flag]
        cache_fig.add_trace(go.Bar(x=[label],y=[float(np.mean(values))],name=label,
            legendgroup=label,marker_color="#d55e00" if not flag else "#0072b2",
            error_y=dict(type="data",array=[max(values)-np.mean(values)],
                         arrayminus=[np.mean(values)-min(values)],symmetric=False),
            showlegend=col==1),1,col)
    cache_fig.update_yaxes(title_text=unit,rangemode="tozero",row=1,col=col)
cache_fig.update_layout(title="캐시 사용 여부 비교: 입력 512token·출력 32token<br><sup>막대: 두 번의 평균 / 선: 관찰 범위</sup>",height=450)
show_save(cache_fig,"03_cache_on_off")
print(CACHE_ROWS)


[{'use_cache': False, 'input_tokens': 512, 'output_tokens': 32, 'seconds': 5.741987510999934, 'peak_gib': 2.9372763633728027}, {'use_cache': True, 'input_tokens': 512, 'output_tokens': 32, 'seconds': 1.253789425999912, 'peak_gib': 2.9441418647766113}, {'use_cache': True, 'input_tokens': 512, 'output_tokens': 32, 'seconds': 1.2717411710000306, 'peak_gib': 2.9441418647766113}, {'use_cache': False, 'input_tokens': 512, 'output_tokens': 32, 'seconds': 5.772982713999909, 'peak_gib': 2.9372763633728027}]


**기록 4:** 네 실행의 출력이 모두 32token인지 확인합니다. 두 패널에서 어느 항목이 달라졌는지 설명하고, 반복 두 번의 차이보다 조건 간 차이가 충분히 큰지도 봅니다. 예상과 다르면 결과를 고치지 않고 실제 설정·예열·측정 범위를 대조합니다.

이 비교의 캐시는 **요청 한 건 안에서 다음 token을 만들 때 재사용하는 KV 캐시**입니다. 서로 다른 질문 사이에 공통 문서를 재사용하는 프리픽스 캐시와 구분합니다. 프리픽스 캐시는 D58에서 다룹니다.


**어텐션 맵 탐색하기**

어텐션 맵은 한 층·한 헤드에서 현재 위치가 앞선 위치의 정보를 섞는 가중치를 색으로 나타낸 그림입니다. **세로 행이 정보를 읽는 쪽, 가로 열이 참고하는 쪽**입니다. 색이 진하다는 사실만으로 답변의 근거·인과관계·사람 같은 이해를 증명하지는 않습니다.

짧은 문장 하나만 관찰합니다. 한국어가 token 조각으로 깨져 보이지 않도록 그림에서는 같은 어절의 질의 행을 평균 내고 키 열을 합칩니다. 따라서 어절로 묶은 가중치 그림이며 원래 token별 행렬과 같은 해상도는 아닙니다.

가중치를 반환받는 이 짧은 실행에서만 어텐션 구현을 `eager`로 바꾸고 곧바로 `sdpa`로 되돌립니다. 그림용 실행 시간을 앞의 성능 비교에 섞지 않습니다. 입력은 96token 이하이며 긴 문서의 어텐션 행렬을 만들지 않습니다.


In [15]:
def word_groups(text,offsets):
    spans=list(re.finditer(r"\S+",text))
    labels=[m.group() for m in spans]
    groups=[]
    for start,end in offsets:
        overlaps=[max(0,min(end,m.end())-max(start,m.start())) for m in spans]
        groups.append(int(np.argmax(overlaps)) if max(overlaps)>0 else
                      min(range(len(spans)),key=lambda j:abs(spans[j].start()-start)))
    return labels,np.asarray(groups)

def aggregate_attention(matrix,groups,n_words):
    membership=np.eye(n_words)[groups]
    counts=membership.sum(axis=0)
    return (membership.T@matrix@membership)/np.maximum(counts[:,None],1)

def collect_attention(text):
    encoded=tok(text,return_offsets_mapping=True,add_special_tokens=False)
    offsets=encoded.pop("offset_mapping")
    if len(encoded["input_ids"])>96:raise ValueError("어텐션 탐색 문장은 96token 이하로 줄이세요.")
    labels,groups=word_groups(text,offsets)
    inputs={k:torch.tensor([v],device=model.device) for k,v in encoded.items()}
    output=None;matrices={}
    try:
        model.set_attn_implementation("eager")
        with torch.inference_mode():
            output=model(**inputs,use_cache=False,output_attentions=True,logits_to_keep=1)
        if output.attentions is None or any(a is None for a in output.attentions):
            raise RuntimeError("어텐션 가중치가 반환되지 않았습니다. 모델과 라이브러리 버전을 확인하세요.")
        layers=sorted({0,len(output.attentions)//2,len(output.attentions)-1})
        for layer in layers:
            for head in range(output.attentions[layer].shape[1]):
                raw=output.attentions[layer][0,head].detach().float().cpu().numpy()
                matrices[f"층 {layer+1} · 헤드 {head+1}"]=aggregate_attention(raw,groups,len(labels))
        return labels,matrices,len(encoded["input_ids"])
    finally:
        output=None
        del inputs
        model.set_attn_implementation("sdpa")
        gc.collect();torch.cuda.empty_cache()

ATTENTION_TEXT="교육장은 오전 9시에 열고 오후 6시에 닫습니다."
word_labels,attention_maps,attention_tokens=collect_attention(ATTENTION_TEXT)
names=list(attention_maps)
attention_fig=go.Figure(go.Heatmap(z=attention_maps[names[0]],x=word_labels,y=word_labels,
    zmin=0,zmax=1,colorscale="Blues",colorbar=dict(title="가중치"),
    hovertemplate="읽는 어절: %{y}<br>참고 어절: %{x}<br>가중치: %{z:.3f}<extra></extra>"))
attention_fig.update_layout(title="어절로 묶은 어텐션: "+names[0],height=560,
    xaxis_title="참고하는 어절",yaxis_title="정보를 읽는 어절",
    yaxis_autorange="reversed",margin=dict(t=110),updatemenus=[dict(
        buttons=[dict(label=name,method="update",args=[{"z":[matrix.tolist()]},
            {"title":{"text":"어절로 묶은 어텐션: "+name}}]) for name,matrix in attention_maps.items()],
        direction="down",x=1,y=1.16,xanchor="right",yanchor="top")])
show_save(attention_fig,"04_attention_explorer")
np.savez_compressed(FIG_DIR/"attention_values.npz",**attention_maps)
RESULTS["attention_probe"]={"text":ATTENTION_TEXT,"input_tokens":attention_tokens,
    "display":"어절별 질의 행 평균·키 열 합산","views":names,
    "backend":"eager; 관찰 후 sdpa 복원","measured":True}


**탐색 과제:** 앞·중간·마지막 층에서 각각 헤드를 골라 `닫습니다.` 행을 살펴봅니다. 같은 어절에서도 값이 어떻게 달라지는지 두 화면을 근거로 씁니다. 미래 어절에 대한 값이 나타나지 않는 부분과, 자기 어절 안의 값이 남는 부분도 구분합니다.

원하는 뜻이 드러나는 헤드 하나만 골라 모델 전체의 판단이라고 설명하지 않습니다. 어절로 묶으면서 같은 어절 안의 token 순서는 그림에서 합쳐졌다는 점도 함께 씁니다.


### Step 5. 메모리 부족 메시지 진단


[목표 2] 아래는 **진단 연습용 예시**입니다. 현재 GPU에서 실제로 발생한 오류가 아닙니다. 요청한 크기와 남은 메모리를 구분하고, 확정한 사실과 추가 확인을 나눠 기록합니다.

이론 5.3의 기존 T4 사례도 함께 읽습니다. 전체 14.56GiB인 장치에서 **추가 텐서 하나에 17.61GiB를 요청했다면**, 빈 캐시 반환만으로 그 요청을 만족시킬 수 없습니다. 아래에서는 해당 요청이 어떤 계산에서 나왔을지 크기로 후보를 좁힙니다.


In [16]:
OOM_EXAMPLE = """CUDA out of memory. Tried to allocate 2.00 GiB. GPU 0 has a total capacity of 14.56 GiB
of which 1.10 GiB is free. Including non-PyTorch memory, this process has 13.45 GiB
memory in use. Of the allocated memory 9.80 GiB is allocated by PyTorch, and 3.20 GiB
is reserved by PyTorch but unallocated."""

def read_oom(message):
    units = {"bytes":1, "KiB":1024, "MiB":1024**2, "GiB":1024**3}
    size = r"([0-9.]+) (bytes|KiB|MiB|GiB)"
    def get(pattern):
        m = re.search(pattern, message)
        return float(m[1])*units[m[2]]/GIB if m else None
    return {"request_gib":get(r"Tried to allocate "+size),
            "total_gib":get(r"total capacity of "+size),
            "free_gib":get(r"of which "+size),
            "allocated_gib":get(r"Of the allocated memory "+size),
            "reserved_unused_gib":get(r"and "+size+r" is reserved by PyTorch but unallocated")}

print(read_oom(OOM_EXAMPLE))
DIAGNOSIS = {"source":"진단 연습용 예시", "observation":"", "check_next":"", "proposed_action":"", "actually_retested":False}
# 빈 항목에 자신의 판단을 적은 뒤 다음 풀이를 여세요.
RESULTS["diagnosis"] = DIAGNOSIS
RESULTS["long"] = {"before_oom":None, "after_gb":None, "status":"긴 입력의 강제 OOM 미실시"}


{'request_gib': 2.0, 'total_gib': 14.56, 'free_gib': 1.1, 'allocated_gib': 9.8, 'reserved_unused_gib': None}


**요청 크기로 계산 후보 찾기**

일부 어텐션 구현이 `[배치, 질의 헤드, 입력 길이, 입력 길이]` 모양의 행렬을 통째로 만들면 입력 길이의 제곱에 비례하는 공간이 필요합니다. 아래는 배치 1·질의 헤드 28·원소당 4byte라는 **가정**을 넣어 요청 크기에 대응하는 길이를 역산합니다. KV 계산의 KV 헤드 수와 다른 값이라는 점도 확인합니다.

계산 결과는 오류 원인의 확정값이 아닙니다. 실제 입력 길이·dtype·어텐션 구현을 확인한 뒤에야 이 후보를 채택할 수 있습니다. 현재 실습의 SDPA 실행이 같은 행렬을 만들었다는 뜻도 아닙니다.


In [17]:
def attention_length_candidate(request_gib, query_heads, element_bytes=4, batch=1):
    return (request_gib*GIB/(batch*query_heads*element_bytes))**0.5

candidate = attention_length_candidate(17.61, query_heads=28)
print(f"기존 사례의 가정에 대응하는 입력 길이: 약 {candidate:,.0f}token")
RESULTS["oom_size_case"] = {
    "source":"기존 T4 오류 사례의 크기를 사용한 계산", "actual_gpu_reproduction":False,
    "request_gib":17.61, "device_total_gib":14.56,
    "assumed_batch":1, "assumed_query_heads":28, "assumed_element_bytes":4,
    "candidate_input_tokens":candidate,
    "conclusion":"입력 길이·dtype·실행 구현 확인이 필요한 후보"}


기존 사례의 가정에 대응하는 입력 길이: 약 12,993token


<details class="lms-answer">
<summary>풀이 보기</summary>

추가 요청 2.00GiB가 현재 남은 1.10GiB보다 크다는 사실을 관찰할 수 있습니다. 예약·미사용 3.20GiB라는 정보만으로 단편화를 단정하지 않습니다. 기존 모델·결과 참조와 메모리 요약을 확인하고, 불필요한 참조를 해제하거나 입력을 줄인 뒤 다시 측정하는 조치를 제안합니다.

예시만 읽었다면 해결 완료로 기록하지 않습니다. 실제 오류를 고쳤다면 변경 전후의 조건과 재실행 결과를 별도로 기록합니다.

역산 결과는 약 13,000token입니다. 입력을 줄이면 이 후보 행렬은 길이의 제곱만큼 줄지만, KV 캐시는 같은 조건에서 길이에 비례해 줄어듭니다. 두 종류의 공간을 구분해야 길이를 줄이는 조치가 어느 부분에 영향을 주는지 설명할 수 있습니다.

이 사례는 현재 GPU에서 재현하지 않았으며 메모리 부족을 강제로 만들지 않습니다. 실행 기록에는 계산과 현재 GPU 실측을 나눠 남깁니다.

</details>


### Step 6. 제공 코드로 가벼운 챗봇 만들기


[목표 3] **이전 파일 없이도 시작할 수 있도록 필요한 프로젝트 파일을 모두 만듭니다.** D52 챗봇을 했다면 파일 이름과 요청 형식이 익숙할 것입니다. 하지 않았다면 아래 순서대로 실행하면 됩니다.

| 유지할 기능 | 오늘 연결할 부분 |
|---|---|
| `/chat`, `messages`, `max_new_tokens`, `temperature` | Qwen의 대화 형식과 Step 3에서 올린 모델 |
| `success`, `response`, `model_name`, `user` 응답 | 생성된 token ID만 잘라 디코딩한 답변 |
| 예시 API 키 인증과 입력 검증 | 전체 대화 token 상한과 모델 한 벌 재사용 |

`bot`으로 받은 이전 답은 모델에 전달할 때 `assistant`로 바꿉니다. 원본의 `GPT2LMHeadModel`에 모델 이름만 바꾸지 않고 `AutoModelForCausalLM`으로 이미 로드한 Qwen을 연결합니다.

서버 파일은 준비 코드입니다. 직접 바꾸어 볼 부분은 대화 내용과 생성 상한이며, 인증·스키마를 처음부터 다시 작성하는 과제는 아닙니다. 파일은 이번 실행의 새 폴더에 만들고 이전 프로젝트는 건드리지 않습니다.


In [18]:
PROJ=WORK/"serving_project"
(PROJ/"app").mkdir(parents=True,exist_ok=True)
(PROJ/"app"/"__init__.py").write_text("",encoding="utf-8")
print("오늘 만들 프로젝트:",PROJ)


오늘 만들 프로젝트: /content/d53/20261105-150928/serving_project


In [19]:
%%writefile serving_project/app/auth.py
import json,os
from fastapi import Header,HTTPException

async def verify_api_key(x_api_key: str | None = Header(default=None)) -> str:
    keys=json.loads(os.environ.get("CHATBOT_API_KEYS",'{"test-key-001":"학습자"}'))
    if not x_api_key or x_api_key not in keys:
        raise HTTPException(status_code=401,detail="유효한 API 키가 필요합니다.")
    return keys[x_api_key]


Writing serving_project/app/auth.py


In [20]:
%%writefile serving_project/app/chatbot_schemas.py
from typing import Literal
from pydantic import BaseModel,Field,model_validator

class Message(BaseModel):
    role: Literal["user","bot","assistant"]
    content: str = Field(...,min_length=1,max_length=1000)

class ChatRequest(BaseModel):
    messages: list[Message] = Field(...,min_length=1)
    max_new_tokens: int = Field(default=100,ge=10,le=500)
    temperature: float = Field(default=0.8,gt=0.0,le=2.0)

    @model_validator(mode="after")
    def last_message_is_user(self):
        if self.messages[-1].role!="user":
            raise ValueError("마지막 메시지는 사용자의 질문이어야 합니다.")
        return self

class ChatResponse(BaseModel):
    success: bool
    response: str
    model_name: str
    user: str | None = None


Writing serving_project/app/chatbot_schemas.py


In [21]:
%%writefile serving_project/app/errors.py
import logging
from fastapi import Request
from fastapi.responses import JSONResponse

class InputTooLongError(ValueError):
    pass

def register_error_handlers(app):
    @app.exception_handler(InputTooLongError)
    async def too_long(request: Request,exc: InputTooLongError):
        return JSONResponse(status_code=413,content={"detail":str(exc)})

    @app.exception_handler(Exception)
    async def unexpected(request: Request,exc: Exception):
        logging.getLogger("chatbot").error("생성 오류",exc_info=True)
        return JSONResponse(status_code=500,content={"detail":"서버 내부 오류가 발생했습니다."})


Writing serving_project/app/errors.py


In [22]:
%%writefile serving_project/app/chatbot_model.py
import threading,time
import torch
from app.errors import InputTooLongError

class ChatbotModel:
    # 새 모델을 다운로드하지 않고 노트북의 모델 한 벌을 연결합니다.
    def __init__(self,tokenizer,model,model_name,max_input_tokens=2048):
        self.tokenizer,self.model=tokenizer,model
        self.model_name=model_name
        self.max_input_tokens=max_input_tokens
        self.lock=threading.Lock()
        self.last_report={}

    def _build_prompt(self,messages):
        chat=[{"role":"system","content":"한국어로 짧게 답하세요. 제공된 안내문에 없는 사실은 확인할 수 없다고 답하세요."}]
        for m in messages:
            chat.append({"role":"assistant" if m["role"]=="bot" else m["role"],"content":m["content"]})
        return self.tokenizer.apply_chat_template(chat,tokenize=False,add_generation_prompt=True)

    def prepare(self,messages,max_new_tokens):
        inputs=self.tokenizer(self._build_prompt(messages),return_tensors="pt")
        n=inputs["input_ids"].shape[1]
        model_limit=getattr(self.model.config,"max_position_embeddings",None)
        if n>self.max_input_tokens or (model_limit and n+max_new_tokens>model_limit):
            raise InputTooLongError(f"입력은 대화 형식을 포함해 {self.max_input_tokens}token 이하여야 합니다.")
        return inputs.to(self.model.device)

    def generate_response(self,messages,max_new_tokens=100,temperature=0.8,top_k=50,top_p=0.9):
        with self.lock:
            inputs=self.prepare(messages,max_new_tokens)
            output=None
            if torch.cuda.is_available():torch.cuda.synchronize()
            started=time.perf_counter()
            try:
                with torch.inference_mode():
                    output=self.model.generate(**inputs,max_new_tokens=max_new_tokens,use_cache=True,
                        do_sample=True,temperature=temperature,top_k=top_k,top_p=top_p,
                        pad_token_id=self.tokenizer.eos_token_id)
                if torch.cuda.is_available():torch.cuda.synchronize()
                n=inputs["input_ids"].shape[1]
                response=self.tokenizer.decode(output[0,n:],skip_special_tokens=True).strip()
                self.last_report={"input_tokens":n,"output_tokens":output.shape[1]-n,
                    "seconds":time.perf_counter()-started,"max_new_tokens":max_new_tokens}
                return response or "(응답이 비어 있습니다)"
            finally:
                output=None
                del inputs

    def release(self):
        with self.lock:self.model=self.tokenizer=None


Writing serving_project/app/chatbot_model.py


In [23]:
%%writefile serving_project/app/model_service.py
import os,time
from app.chatbot_model import ChatbotModel

MODEL_VERSION=os.environ.get("MODEL_VERSION","d53-unset")
_STATE={"chatbot":None}

def attach(tokenizer,model,model_name):
    _STATE["chatbot"]=ChatbotModel(tokenizer,model,model_name)
    return _STATE["chatbot"]

def generate_reply(messages,max_new_tokens=100,temperature=0.8):
    if messages[-1]["content"]=="__TRIGGER_500_ERROR__":
        raise RuntimeError("test error at /app/secret_path/model.py")
    chatbot=_STATE["chatbot"]
    if chatbot is None:raise RuntimeError("노트북의 모델을 먼저 연결하세요.")
    started=time.perf_counter()
    text=chatbot.generate_response(messages,max_new_tokens,temperature)
    return {"response":text,"turn_count":(len(messages)+1)//2,
            "elapsed_ms":(time.perf_counter()-started)*1000,"model_version":MODEL_VERSION}

def unload():
    if _STATE["chatbot"] is not None:_STATE["chatbot"].release()
    _STATE["chatbot"]=None


Writing serving_project/app/model_service.py


In [24]:
%%writefile serving_project/app/chatbot_api.py
import asyncio
from contextlib import asynccontextmanager
from concurrent.futures import ThreadPoolExecutor
from fastapi import Depends,FastAPI
from app.auth import verify_api_key
from app.chatbot_schemas import ChatRequest,ChatResponse
from app.errors import register_error_handlers
from app.model_service import MODEL_VERSION,generate_reply

@asynccontextmanager
async def lifespan(app):
    app.state.executor=ThreadPoolExecutor(max_workers=1)
    try:yield
    finally:app.state.executor.shutdown(wait=True)

app=FastAPI(title="D53 Chatbot",version=MODEL_VERSION,lifespan=lifespan)
register_error_handlers(app)

@app.get("/health")
async def health():
    return {"status":"healthy","model":MODEL_VERSION}

@app.post("/chat",response_model=ChatResponse)
async def chat(req: ChatRequest,user: str = Depends(verify_api_key)):
    messages=[m.model_dump() for m in req.messages]
    result=await asyncio.get_running_loop().run_in_executor(
        app.state.executor,generate_reply,messages,req.max_new_tokens,req.temperature)
    return ChatResponse(success=True,response=result["response"],
                        model_name=result["model_version"].split("@")[0],user=user)


Writing serving_project/app/chatbot_api.py


In [25]:
%%writefile serving_project/app/main.py
# uvicorn app.main:app으로도 실행할 수 있는 진입점입니다.
from app.chatbot_api import app


Writing serving_project/app/main.py


**연결 확인:** `chatbot_model.py`는 대화 형식·token 수·생성을 맡고, `chatbot_api.py`는 인증된 요청과 응답 형식을 맡습니다. 내부 `generate_reply()`의 측정용 값과 바깥 `/chat`의 D52 응답 필드는 서로 다릅니다. API 응답을 읽을 때는 위 표의 네 필드를 확인합니다.

다음 셀은 준비한 모델을 연결하고 서버가 실제로 시작했는지 기다립니다. 같은 GPU에 서버용 모델을 한 벌 더 올리지 않습니다. API 주소는 같은 Colab 런타임 안에서만 사용합니다.


In [26]:
import socket,requests,uvicorn
os.environ.update(MODEL_VERSION=f"{REPO}@{REVISION[:12]}",
                  CHATBOT_API_KEYS=json.dumps({"test-key-001":"학습자"},ensure_ascii=False))
if "app" in sys.modules:raise RuntimeError("app 모듈이 이미 있습니다. 새 세션에서 순서대로 실행하세요.")
sys.path.insert(0,str(PROJ))
from app import model_service as SERVICE
CHATBOT=SERVICE.attach(tok,model,REPO)
from app.chatbot_api import app
def available_port():
    for port in range(8000,8011):
        with socket.socket() as sock:
            try:sock.bind(("127.0.0.1",port));return port
            except OSError:pass
    raise RuntimeError("8000~8010 포트가 사용 중입니다. 이전 서버를 확인하세요.")
PORT=available_port();API_BASE=f"http://127.0.0.1:{PORT}"
server=uvicorn.Server(uvicorn.Config(app,host="127.0.0.1",port=PORT,loop="asyncio",log_level="warning"))
server_thread=threading.Thread(target=server.run,daemon=True)
server_thread.start()
for _ in range(100):
    if server.started:break
    if not server_thread.is_alive():raise RuntimeError("서버 시작에 실패했습니다. 위 오류를 확인하세요.")
    time.sleep(0.1)
if not server.started:raise RuntimeError("서버 시작 대기 시간이 지났습니다.")
print(requests.get(API_BASE+"/health",timeout=10).json())


{'status': 'healthy', 'model': 'Qwen/Qwen2.5-1.5B-Instruct@989aa7980e4c'}


In [27]:
HEADERS={"X-API-Key":"test-key-001"}
history=[]
def chat_turn(text,reset=False):
    if reset:history.clear()
    sent=history+[{"role":"user","content":text}]
    r=requests.post(API_BASE+"/chat",json={"messages":sent,"max_new_tokens":64,"temperature":0.3},
                    headers=HEADERS,timeout=120)
    r.raise_for_status();body=r.json()
    history.extend([{"role":"user","content":text},{"role":"bot","content":body["response"]}])
    print("나:",text)
    print("챗봇:",body["response"])
    return body

API_RESULT=chat_turn(DOC+"\n"+QUESTION,reset=True)
chat_turn("앞서 안내한 교육장은 몇 시에 문을 닫습니까?")
RESULTS["api_response"]=API_RESULT
RESULTS["chatbot_generation"]=dict(CHATBOT.last_report)


나: 실습용 안내문: 교육장 이용 시간은 오전 9시부터 오후 6시까지입니다.
교육장은 몇 시에 문을 닫습니까?
챗봇: 교육장의 문은 오후 6시에 닫습니다.
나: 앞서 안내한 교육장은 몇 시에 문을 닫습니까?
챗봇: 교육장은 오후 6시에 문을 닫습니다.


이 셀의 `chat_turn("질문")`을 다시 호출하면 이전 대화를 함께 보냅니다. `reset=True`로 새 대화를 시작할 수 있습니다. 답의 내용은 안내문과 대조하며, 형식 검사가 통과했다고 내용도 정확하다고 판단하지 않습니다.


아래 셀 실행 시, 검사 중 의도적인 오류 로그가 출력됩니다.

In [28]:
def post_chat(body,headers=None):
    return requests.post(API_BASE+"/chat",json=body,headers=HEADERS if headers is None else headers,timeout=120)
one={"messages":[{"role":"user","content":"한국어로 짧게 인사해 주세요."}],"max_new_tokens":16}
normal=post_chat(one)
fault=post_chat({"messages":[{"role":"user","content":"__TRIGGER_500_ERROR__"}]})
long_body={"messages":[{"role":"user","content":"참고 자료입니다. "*80} for _ in range(20)]}
GRADE={"정상 응답":normal.status_code==200 and set(normal.json())=={"success","response","model_name","user"},
       "인증 누락":post_chat(one,{}).status_code==401,
       "잘못된 키":post_chat(one,{"X-API-Key":"wrong-key"}).status_code==401,
       "빈 대화":post_chat({"messages":[]}).status_code==422,
       "생성 설정 범위":post_chat({**one,"temperature":5}).status_code==422,
       "전체 입력 상한":post_chat(long_body).status_code==413,
       "내부 오류 비노출":fault.status_code==500 and "secret_path" not in fault.text and "Traceback" not in fault.text}
for name,ok in GRADE.items():print("PASS" if ok else "FAIL",name)
RESULTS["grade"]=GRADE


ERROR:chatbot:생성 오류
Traceback (most recent call last):
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/errors.py", line 165, in __call__
    await self.app(scope, receive, _send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/middleware/exceptions.py", line 62, in __call__
    await wrap_app_handling_exceptions(self.app, conn)(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/_exception_handler.py", line 62, in wrapped_app
    raise exc
  File "/usr/local/lib/python3.13/dist-packages/starlette/_exception_handler.py", line 51, in wrapped_app
    await app(scope, receive, sender)
  File "/usr/local/lib/python3.13/dist-packages/starlette/routing.py", line 715, in __call__
    await self.middleware_stack(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/routing.py", line 735, in app
    await route.handle(scope, receive, send)
  File "/usr/local/lib/python3.13/dist-packages/starlette/routing.py", li

PASS 정상 응답
PASS 인증 누락
PASS 잘못된 키
PASS 빈 대화
PASS 생성 설정 범위
PASS 전체 입력 상한
PASS 내부 오류 비노출


**기록 5:** 요청에서 대화가 어디에 들어가고, API가 어떤 네 필드를 돌려줬는지 표시합니다. 생성 함수의 token 수와 실제 답을 함께 읽습니다. 위 검사는 기존 기능이 유지되는지 한 번 확인하는 회귀 검사이며 오늘의 새 성취를 대신하지 않습니다.


### Step 7. 출력 소비 중단과 모델 생성 중단 비교


[목표 3] **답변을 읽는 반복을 멈추는 것과 모델의 계산을 멈추는 것은 다른 동작입니다.** 모델을 별도 작업에서 돌리면 읽기를 중단해도 생성은 이어질 수 있습니다. 이번에는 같은 계산에서 두 경우를 직접 비교합니다.

준비 코드는 새로 생성된 token ID를 큐에 넣습니다. 큐는 두 작업 사이에서 값을 전달하는 통로입니다. 노트북은 8token을 읽은 뒤 소비를 멈추고, 한 경우에만 생성 중지 신호도 보냅니다. `StoppingCriteria`는 다음 생성 단계에서 그 신호를 확인하는 조건입니다.

출력 상한은 두 경우 모두 96token이고, 자연 종료 차이를 줄이려고 이 실험에서만 최소 길이도 96으로 맞춥니다. 별도 중지 조건은 그 전에 계산을 끝낼 수 있습니다. **API에서 연결을 끊는 시험이 아니라 생성 작업의 중단을 분리해 보는 실험**입니다.


In [29]:
import queue
from transformers import StoppingCriteria,StoppingCriteriaList

class StopOnSignal(StoppingCriteria):
    def __init__(self,event):self.event=event
    def __call__(self,input_ids,scores,**kwargs):
        return torch.full((input_ids.shape[0],),self.event.is_set(),dtype=torch.bool,device=input_ids.device)

class TokenProbe:
    def __init__(self):
        self.q=queue.Queue();self.skip_prompt=True;self.count=0;self.trace=[];self.started=0
    def put(self,value):
        if self.skip_prompt:self.skip_prompt=False;return
        for token_id in value.reshape(-1).tolist():
            self.count+=1
            self.trace.append((time.perf_counter()-self.started,self.count))
            self.q.put(token_id)
    def end(self):self.q.put(None)

def run_stop_case(send_stop,limit=96,read_until=8):
    stop=threading.Event();probe=TokenProbe();result={};errors=[]
    def worker():
        inputs=output=None
        try:
            with CHATBOT.lock:
                inputs=fixed_length_inputs(256)
                torch.cuda.reset_peak_memory_stats(0);torch.cuda.synchronize()
                probe.started=time.perf_counter()
                with torch.inference_mode():
                    output=model.generate(**inputs,min_new_tokens=limit,max_new_tokens=limit,
                        do_sample=False,use_cache=True,pad_token_id=tok.eos_token_id,
                        streamer=probe,stopping_criteria=StoppingCriteriaList([StopOnSignal(stop)]))
                torch.cuda.synchronize()
                result.update(output_tokens=output.shape[1]-inputs["input_ids"].shape[1],
                    seconds=time.perf_counter()-probe.started,peak_gib=torch.cuda.max_memory_allocated(0)/GIB)
        except Exception as exc:errors.append(str(exc))
        finally:
            inputs=output=None
            probe.end()
    worker_thread=threading.Thread(target=worker,daemon=True)
    worker_thread.start()
    read=0;cut_time=None;produced_at_cut=None
    try:
        while True:
            value=probe.q.get(timeout=60)
            if value is None:break
            read+=1
            if read>=read_until:
                cut_time=time.perf_counter()-probe.started
                produced_at_cut=probe.count
                if send_stop:stop.set()
                break
        worker_thread.join(timeout=60)
        if worker_thread.is_alive():raise RuntimeError("생성 작업이 끝나지 않았습니다. 다음 비교를 진행하지 마세요.")
        if errors:raise RuntimeError(errors[0])
        result.update(send_stop=send_stop,read_tokens=read,consumer_stop_seconds=cut_time,
            produced_at_consumer_stop=produced_at_cut,trace=probe.trace,
            worker_finished=True,limit=limit)
        return result
    finally:
        stop.set()
        if worker_thread.is_alive():worker_thread.join(timeout=10)

STOP_ROWS=[run_stop_case(False),run_stop_case(True)]
RESULTS["stop_cases"]=STOP_ROWS
stop_fig=go.Figure()
for row,label,color in zip(STOP_ROWS,["읽기만 중단","생성 중지 신호도 전달"],["#d55e00","#0072b2"]):
    points=[(0,0)]+row["trace"]
    stop_fig.add_trace(go.Scatter(x=[p[0] for p in points],y=[p[1] for p in points],
        mode="lines",name=label,line=dict(color=color)))
    if row["consumer_stop_seconds"] is not None:
        stop_fig.add_trace(go.Scatter(x=[row["consumer_stop_seconds"]],
            y=[row["produced_at_consumer_stop"]],mode="markers",name=label+"의 소비 중단",
            marker=dict(symbol="x",size=13,color=color)))
stop_fig.update_layout(title="읽기를 멈춘 뒤에도 모델이 계산했는지 확인합니다",
    xaxis_title="생성 시작 후 초",yaxis_title="실제로 생성한 token 수",height=440)
show_save(stop_fig,"05_stop_generation")
print([{k:v for k,v in r.items() if k!="trace"} for r in STOP_ROWS])


[{'output_tokens': 96, 'seconds': 3.6078384470001765, 'peak_gib': 2.925210475921631, 'send_stop': False, 'read_tokens': 8, 'consumer_stop_seconds': 0.348864801000218, 'produced_at_consumer_stop': 8, 'worker_finished': True, 'limit': 96}, {'output_tokens': 9, 'seconds': 0.33842068300009487, 'peak_gib': 2.925210475921631, 'send_stop': True, 'read_tokens': 8, 'consumer_stop_seconds': 0.30167857200012804, 'produced_at_consumer_stop': 8, 'worker_finished': True, 'limit': 96}]


**기록 6:** X 표시는 노트북이 읽기를 멈춘 시점입니다. 그 뒤 선이 계속 올라가는지, 실제 생성한 token 수와 종료 시간이 어떻게 다른지 설명합니다. 중지 신호는 이미 수행한 계산을 되돌리지 않고 다음 생성 단계에서 확인됩니다.

빠른 GPU에서는 신호를 보내기 전에 생성이 끝날 수도 있습니다. 이때는 효과를 확인하지 못한 것으로 기록합니다. 두 결과를 원하는 값으로 고치지 않습니다. 큐에서 읽은 횟수와 그 시점까지 생성한 token 수도 같다고 가정하지 않습니다.

모델은 메모리에 계속 올라 있으므로 생성을 멈춘 뒤 전체 사용량이 0이 되어야 하는 것은 아닙니다. 이 실험만으로 HTTP 취소 처리나 여러 사용자의 서비스 용량까지 검증했다고 쓰지 않습니다.


### Step 8. 그림·코드·결과 보관과 런타임 해제


그림은 실행 중의 실제 값으로 만들고 HTML로 보관합니다. 내려받은 압축을 풀면 인터넷 연결 없이도 그림의 확대·범례·어텐션 메뉴를 사용할 수 있습니다. `figures` 폴더의 `plotly.min.js`도 함께 둡니다.

아래 셀은 결과와 API 코드를 보관하고 오늘 시작한 서버와 모델을 해제합니다. 원본 파일과 모델 가중치는 삭제하지 않습니다. **PC에서 결과 압축을 열어 확인한 뒤 Colab 화면에서 런타임을 해제합니다.**


In [30]:
RESULTS["model"].update(precision="fp16",parameters=PARAMS)
RESULTS["execution_status"]="실제 GPU의 실험 셀 실행을 마침"
RESULTS["measurement_note"]="1.5B fp16 실험. 모의 OOM 사례 및 어텐션 관찰용 eager 실행은 성능 측정과 분리함."
RESULTS["bandwidth_gbs"]=None;RESULTS["tps"]=None
project_zip=shutil.make_archive(str(RESULT_DIR/"d53_chatbot_project"),"zip",root_dir=WORK,base_dir="serving_project")
RESULTS["handoff"]={"project_archive":project_zip,"source_platform":"Colab",
    "chat_response_fields":["success","response","model_name","user"],
    "generation_function":"generate_reply(messages,max_new_tokens,temperature)",
    "d57_requires_response_check_update":True}
gpu_tag=re.sub(r"[^A-Za-z0-9]+","_",GPU_NAME).strip("_")
out_path=RESULT_DIR/f"results_{gpu_tag}.json"
out_path.write_text(json.dumps(RESULTS,ensure_ascii=False,indent=2),encoding="utf-8")
server.should_exit=True;server_thread.join(timeout=15)
if server_thread.is_alive():raise RuntimeError("서버가 종료되지 않았습니다. 진행 중인 요청을 확인하세요.")
SERVICE.unload();CHATBOT=None
del model
gc.collect();torch.cuda.empty_cache()
print("저장한 결과:",out_path,"그림:",FIG_DIR)
print("해제 후 상태:",snapshot("종료 후"))


저장한 결과: /content/results/d53/20261105-150928/results_Tesla_T4.json 그림: /content/results/d53/20261105-150928/figures
해제 후 상태: {'label': '종료 후', 'free_gib': 11.53497314453125, 'total_gib': 14.56317138671875, 'allocated_gib': 0.0267333984375, 'reserved_gib': 2.880859375, 'peak_gib': 2.925210475921631}


In [31]:
from google.colab import files

# 압축 파일은 결과 폴더 바깥에 만들어 자기 자신이 다시 포함되지 않게 합니다.
bundle_path = Path(shutil.make_archive(
    str(RESULT_DIR.parent/f"d53_{RUN_ID}_submission"), "zip", root_dir=RESULT_DIR))
print("내려받을 결과 묶음:",bundle_path)
files.download(str(bundle_path))


내려받을 결과 묶음: /content/results/d53/d53_20261105-150928_submission.zip


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

**마지막 확인**

1. PC에서 `d53_<실행 시각>_submission.zip`을 직접 엽니다.
2. 측정 JSON, `model_manifest.json`, `d53_chatbot_project.zip`, `figures` 폴더를 확인합니다.
3. 압축을 모두 푼 뒤 HTML 그림 하나와 어텐션 메뉴를 열어 봅니다. 다운로드가 막히면 Colab 파일 패널에서 압축을 직접 내려받습니다.
4. 노트북 자체도 Drive에 저장됐는지 확인합니다. 결과 압축과 노트북은 다른 파일입니다.
5. 보관을 확인한 뒤 **연결 해제 및 런타임 삭제**로 마칩니다. 탭만 닫는 것과 구분합니다.


### 마무리 과제. 그림으로 실행 조건 설명하기


각자 자기 결과에서 그림 세 곳을 골라 처음 적은 예측을 고칩니다. 그림을 붙이기만 하지 않고 관찰한 값과 결론의 범위를 함께 적습니다.

| 목표 | 제출할 증거 |
|---|---|
| 1 | GPU·모델 해시·fp16 기록과 가중치/KV/최대값을 구분한 그림 해석 |
| 2 | 같은 길이에서 캐시를 켜고 끈 결과, 어텐션 두 화면의 차이와 해석의 한계 |
| 3 | 챗봇의 두 차례 대화·응답 네 필드, 소비 중단 뒤 생성 곡선과 작업 종료 여부 |

**나가기 질문:** 긴 문서를 받는 챗봇에서 메모리를 아끼려고 캐시를 끄고, 사용자가 떠나면 출력만 멈추자는 제안을 받았습니다. 오늘의 그림 두 개 이상으로 어떤 부분을 확인하고 어떻게 수정할지 설명합니다.

| 스스로 점검할 내용 | 확인 / 다시 볼 단계 |
|---|---|
| 모델 자체와 요청이 쓰는 메모리의 측정 시점을 구분했습니다 | |
| 어텐션 그림의 색을 답변의 인과적 근거로 단정하지 않았습니다 | |
| 읽기 중단·중지 신호·생성 종료가 다른 사건임을 설명했습니다 | |


### Reference

- [Colab 공식 FAQ](https://research.google.com/colaboratory/faq.html)
- [Qwen2.5-1.5B-Instruct 설정](https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct/blob/main/config.json)
- [Transformers의 KV 캐시](https://huggingface.co/docs/transformers/v5.17.0/en/cache_explanation)
- [어텐션 구현 전환](https://huggingface.co/docs/transformers/v5.17.0/en/attention_interface)
- [생성 중지 조건과 스트리머](https://huggingface.co/docs/transformers/v5.17.0/en/internal/generation_utils)
- [Plotly HTML 내보내기](https://plotly.com/python/interactive-html-export/)
- [어텐션 해석의 한계 연구: Jain·Wallace, 2019](https://aclanthology.org/N19-1357/)

실제 GPU·측정값·그림은 각자의 실행 결과에 남깁니다.
